# BiLSTM-CRF for ABSA

Improvements: NFC Unicode normalization, span whitespace trimming, BILOU tagging, sqrt-balanced class weights, span-based evaluation.

In [12]:
!pip install pytorch-crf seqeval transformers

In [13]:
import json
import re
import unicodedata
import os
from collections import defaultdict, Counter
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, set_seed
from torchcrf import CRF
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score

In [14]:
# --- DATA PREPARATION ---
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

@dataclass(frozen=True)
class SpanLabel:
    start: int
    end: int
    label: str

    def clamped(self, text: str):
        start = max(0, min(self.start, len(text)))
        end = max(start, min(self.end, len(text)))
        return SpanLabel(start, end, self.label)

@dataclass(frozen=True)
class Example:
    text: str
    labels: list

def normalize_nfc_with_offsets(text, raw_labels):
    """Normalize text to NFC and realign character-level span offsets."""
    nfc_text = unicodedata.normalize("NFC", text)
    if text == nfc_text:
        return text, raw_labels
    new_labels = []
    for start, end, label in raw_labels:
        new_start = len(unicodedata.normalize("NFC", text[:start]))
        new_end = len(unicodedata.normalize("NFC", text[:end]))
        new_labels.append((new_start, new_end, label))
    return nfc_text, new_labels

def read_jsonl(path):
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            text = str(row["text"])
            raw_labels = [(int(a), int(b), str(l)) for a, b, l in row.get("labels", [])]
            # NFC Unicode normalization + offset realignment
            text, raw_labels = normalize_nfc_with_offsets(text, raw_labels)
            # Trim leading/trailing whitespace from span boundaries
            trimmed = []
            for s, e, l in raw_labels:
                span_text = text[s:e]
                leading = len(span_text) - len(span_text.lstrip())
                trailing = len(span_text) - len(span_text.rstrip())
                ns, ne = s + leading, e - trailing
                if ns < ne:
                    trimmed.append((ns, ne, l))
            raw_labels = trimmed
            labels = [SpanLabel(s, e, l) for s, e, l in raw_labels]
            examples.append(Example(text=text, labels=labels))
    return examples

def iter_token_spans(text: str):
    return [(m.start(), m.end(), m.group(0)) for m in TOKEN_RE.finditer(text)]

def tag_prefix(pos: int, length: int, scheme: str) -> str:
    if scheme == "bio":
        return "B" if pos == 0 else "I"
    elif scheme == "bilou":
        if length == 1:
            return "U"
        elif pos == 0:
            return "B"
        elif pos == length - 1:
            return "L"
        else:
            return "I"
    return ""

def build_sequence_labels(span_labels: Iterable[str], scheme: str = "bio") -> list:
    prefixes = ["B", "I"] if scheme == "bio" else ["B", "I", "L", "U"]
    labels = ["O"]
    for label in sorted(span_labels):
        for prefix in prefixes:
            labels.append(f"{prefix}-{label}")
    return labels

def bio_tags_to_spans(tag_ids, token_spans, id2label):
    """Convert BIO/BILOU tag ID sequence to character-level spans.

    Args:
        tag_ids: list of int, tag IDs for each word (-100 = padding)
        token_spans: list of (char_start, char_end, word_text) per word
        id2label: dict mapping tag ID to tag string

    Returns:
        set of (start_char, end_char, label) tuples
    """
    spans = set()
    current_label = None
    current_start = None
    current_end = None

    for i, tag_id in enumerate(tag_ids):
        if tag_id == -100 or i >= len(token_spans):
            break
        tag = id2label.get(tag_id, "O")

        if tag.startswith("U-"):
            # Unit span (single token)
            if current_label is not None:
                spans.add((current_start, current_end, current_label))
                current_label = None
            label = tag[2:]
            spans.add((token_spans[i][0], token_spans[i][1], label))
        elif tag.startswith("B-"):
            if current_label is not None:
                spans.add((current_start, current_end, current_label))
            current_label = tag[2:]
            current_start = token_spans[i][0]
            current_end = token_spans[i][1]
        elif tag.startswith("I-"):
            label = tag[2:]
            if current_label == label:
                current_end = token_spans[i][1]
            else:
                if current_label is not None:
                    spans.add((current_start, current_end, current_label))
                current_label = label
                current_start = token_spans[i][0]
                current_end = token_spans[i][1]
        elif tag.startswith("L-"):
            label = tag[2:]
            if current_label == label:
                current_end = token_spans[i][1]
                spans.add((current_start, current_end, current_label))
                current_label = None
            else:
                if current_label is not None:
                    spans.add((current_start, current_end, current_label))
                # Orphan L-, treat as unit span
                spans.add((token_spans[i][0], token_spans[i][1], label))
                current_label = None
        else:  # O tag
            if current_label is not None:
                spans.add((current_start, current_end, current_label))
                current_label = None

    if current_label is not None:
        spans.add((current_start, current_end, current_label))

    return spans

def compute_span_metrics(examples, pred_tag_seqs, token_spans_list, id2label):
    """Compute span-level micro Precision, Recall, F1 (exact match)."""
    total_tp = 0
    total_pred = 0
    total_gold = 0

    for ex, pred_tags, tok_spans in zip(examples, pred_tag_seqs, token_spans_list):
        gold_spans = {(l.start, l.end, l.label) for l in ex.labels}
        pred_spans = bio_tags_to_spans(pred_tags, tok_spans, id2label)

        total_tp += len(gold_spans & pred_spans)
        total_pred += len(pred_spans)
        total_gold += len(gold_spans)

    precision = total_tp / total_pred if total_pred > 0 else 0.0
    recall = total_tp / total_gold if total_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"precision": precision, "recall": recall, "f1": f1}


def write_jsonl(path, examples):
    rows = []
    for ex in examples:
        labels = [[l.start, l.end, l.label] for l in ex.labels]
        rows.append({"text": ex.text, "labels": labels})
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [15]:
# --- VOCAB & DATASET ---
class Vocab:
    def __init__(self, pad_token="<PAD>", unk_token="<UNK>"):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.token2id = {pad_token: 0, unk_token: 1}
        self.id2token = {0: pad_token, 1: unk_token}
    
    def add(self, token):
        if token not in self.token2id:
            idx = len(self.token2id)
            self.token2id[token] = idx
            self.id2token[idx] = token
            
    def get_id(self, token):
        return self.token2id.get(token, self.token2id[self.unk_token])
    
    def __len__(self):
        return len(self.token2id)

def build_vocabs(examples):
    word_vocab = Vocab()
    char_vocab = Vocab()
    for ex in examples:
        for _, _, word in iter_token_spans(ex.text):
            word_vocab.add(word.lower())
            for c in word:
                char_vocab.add(c)
    return word_vocab, char_vocab

class BiLSTMCRFDataset(Dataset):
    def __init__(self, examples, word_vocab, char_vocab, label2id, tokenizer, tag_scheme="bilou", is_train=True):
        self.examples = examples
        self.word_vocab = word_vocab
        self.char_vocab = char_vocab
        self.label2id = label2id
        self.tokenizer = tokenizer
        self.tag_scheme = tag_scheme
        self.all_token_spans = []  # Store for span-based evaluation
        self.features = self._process_examples()

    def _process_examples(self):
        features = []
        for ex in self.examples:
            spans = iter_token_spans(ex.text)
            self.all_token_spans.append(spans)  # Save for span evaluation
            words = [text for _, _, text in spans]
            
            subword_tokens = self.tokenizer(
                words, 
                is_split_into_words=True,
                add_special_tokens=True,
                return_offsets_mapping=False
            )
            input_ids = subword_tokens["input_ids"]
            attention_mask = subword_tokens["attention_mask"]
            word_ids_hf = subword_tokens.word_ids()
            
            first_subword_mask = [-1] * len(words)
            for idx, w_id in enumerate(word_ids_hf):
                if w_id is not None and first_subword_mask[w_id] == -1:
                    first_subword_mask[w_id] = idx
            
            for w in range(len(words)):
                if first_subword_mask[w] == -1:
                    first_subword_mask[w] = 0
            
            word_ids = [self.word_vocab.get_id(w.lower()) for w in words]
            char_ids = [[self.char_vocab.get_id(c) for c in w] for w in words]
            word_lengths = [len(c) for c in char_ids]
            
            labels = [self.label2id["O"]] * len(words)
            for span in [label.clamped(ex.text) for label in ex.labels]:
                if span.start >= span.end: continue
                covered = [
                    i for i, (start, end, _) in enumerate(spans)
                    if not (start == 0 and end == 0) and start < span.end and end > span.start
                ]
                for pos, token_idx in enumerate(covered):
                    prefix = tag_prefix(pos, len(covered), self.tag_scheme)
                    label_str = f"{prefix}-{span.label}"
                    if label_str in self.label2id:
                        labels[token_idx] = self.label2id[label_str]
            
            features.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "word_ids": word_ids,
                "char_ids": char_ids,
                "word_lengths": word_lengths,
                "first_subword_mask": first_subword_mask,
                "labels": labels
            })
        return features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

def collate_fn(batch):
    max_subwords = max(len(x["input_ids"]) for x in batch)
    max_words = max(len(x["word_ids"]) for x in batch)
    max_word_len = max(max(x["word_lengths"]) if x["word_lengths"] else 0 for x in batch)
    
    if max_words == 0: max_words = 1
    if max_word_len == 0: max_word_len = 1
    batch_size = len(batch)
    
    input_ids = torch.zeros((batch_size, max_subwords), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_subwords), dtype=torch.long)
    word_ids = torch.zeros((batch_size, max_words), dtype=torch.long)
    char_ids = torch.zeros((batch_size, max_words, max_word_len), dtype=torch.long)
    word_lengths = torch.zeros((batch_size, max_words), dtype=torch.long)
    first_subword_mask = torch.zeros((batch_size, max_words), dtype=torch.long)
    labels = torch.full((batch_size, max_words), -100, dtype=torch.long)
    
    for i, ex in enumerate(batch):
        sub_len = len(ex["input_ids"])
        input_ids[i, :sub_len] = torch.tensor(ex["input_ids"])
        attention_mask[i, :sub_len] = torch.tensor(ex["attention_mask"])
        
        word_len = len(ex["word_ids"])
        word_ids[i, :word_len] = torch.tensor(ex["word_ids"])
        word_lengths[i, :word_len] = torch.tensor(ex["word_lengths"])
        first_subword_mask[i, :word_len] = torch.tensor(ex["first_subword_mask"])
        labels[i, :word_len] = torch.tensor(ex["labels"])
        
        for w, chars in enumerate(ex["char_ids"]):
            c_len = len(chars)
            char_ids[i, w, :c_len] = torch.tensor(chars)
            
    return {
        "input_ids": input_ids, "attention_mask": attention_mask,
        "word_ids": word_ids, "char_ids": char_ids,
        "word_lengths": word_lengths, "first_subword_mask": first_subword_mask,
        "tags": labels
    }


In [16]:
# --- MODEL ---
class CharLSTM(nn.Module):
    def __init__(self, char_vocab_size: int, char_emb_dim: int = 50, hidden_dim: int = 50):
        super().__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(char_emb_dim, hidden_dim, bidirectional=True, batch_first=True)

    def forward(self, char_ids: torch.Tensor, word_lengths: torch.Tensor):
        batch_size, seq_len, max_word_len = char_ids.size()
        char_ids_flat = char_ids.view(-1, max_word_len)
        word_lengths_flat = word_lengths.view(-1).cpu()
        
        zero_length_mask = word_lengths_flat == 0
        word_lengths_flat[zero_length_mask] = 1
        
        embeds = self.char_embedding(char_ids_flat)
        lengths, sort_idx = word_lengths_flat.sort(0, descending=True)
        embeds_sorted = embeds[sort_idx]
        
        packed = nn.utils.rnn.pack_padded_sequence(embeds_sorted, lengths, batch_first=True)
        _, (hidden, _) = self.lstm(packed)
        char_repr = torch.cat((hidden[0], hidden[1]), dim=-1)
        
        _, unsort_idx = sort_idx.sort(0)
        char_repr = char_repr[unsort_idx]
        char_repr[zero_length_mask] = 0.0
        
        return char_repr.view(batch_size, seq_len, -1)

class BiLSTM_CRF_ABSA(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_labels, transformer_name="FacebookAI/xlm-roberta-base",
                 syllable_emb_dim=100, lstm_hidden_size=400, dropout_rate=0.33, freeze_transformer=False):
        super().__init__()
        self.num_labels = num_labels
        
        self.transformer = AutoModel.from_pretrained(transformer_name)
        if freeze_transformer:
            for param in self.transformer.parameters():
                param.requires_grad = False
        transformer_hidden_size = self.transformer.config.hidden_size
        
        self.syllable_embedding = nn.Embedding(vocab_size, syllable_emb_dim, padding_idx=0)
        self.char_lstm = CharLSTM(char_vocab_size, char_emb_dim=50, hidden_dim=50)
        
        total_emb_dim = transformer_hidden_size + syllable_emb_dim + 100
        self.dropout = nn.Dropout(dropout_rate)
        self.bilstm = nn.LSTM(total_emb_dim, lstm_hidden_size, num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(lstm_hidden_size * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        
    def _get_lstm_features(self, input_ids, attention_mask, word_ids, char_ids, word_lengths, first_subword_mask):
        transformer_outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        subword_embeddings = transformer_outputs.last_hidden_state 
        
        batch_size = input_ids.size(0)
        max_words = word_ids.size(1)
        contextual_embeds = torch.zeros(batch_size, max_words, subword_embeddings.size(-1), device=input_ids.device)
        for b in range(batch_size):
            valid_subwords = subword_embeddings[b][first_subword_mask[b]]
            length = min(max_words, valid_subwords.size(0))
            contextual_embeds[b, :length] = valid_subwords[:length]
            
        syllable_embeds = self.syllable_embedding(word_ids)
        char_embeds = self.char_lstm(char_ids, word_lengths)
        
        combined_embeds = torch.cat([contextual_embeds, syllable_embeds, char_embeds], dim=-1)
        combined_embeds = self.dropout(combined_embeds)
        
        word_mask = (word_ids != 0)
        word_lengths_list = word_mask.sum(dim=1).cpu()
        word_lengths_list[word_lengths_list == 0] = 1 
        
        packed_embeds = nn.utils.rnn.pack_padded_sequence(combined_embeds, word_lengths_list, batch_first=True, enforce_sorted=False)
        packed_lstm_out, _ = self.bilstm(packed_embeds)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_lstm_out, batch_first=True, total_length=max_words)
        
        lstm_out = self.dropout(lstm_out)
        lstm_feats = self.hidden2tag(lstm_out)
        return lstm_feats, word_mask

    def forward(self, input_ids, attention_mask, word_ids, char_ids, word_lengths, first_subword_mask, tags=None):
        lstm_feats, word_mask = self._get_lstm_features(input_ids, attention_mask, word_ids, char_ids, word_lengths, first_subword_mask)
        if tags is not None:
            # Note: Do not scale lstm_feats with class_weights here. It destabilizes CRF transitions.
            loss = -self.crf(lstm_feats, tags, mask=word_mask, reduction='mean')
            return loss
        else:
            return self.crf.decode(lstm_feats, mask=word_mask)


In [17]:
# --- CONFIGURATION ---
# IMPORTANT: Update DATA_DIR to match your Kaggle path
DATA_DIR = "/kaggle/input/datasets/chunlz/nlp-project/data"
OUTPUT_DIR = "outputs/bilstm-crf"

MODEL_NAME = "FacebookAI/xlm-roberta-base"
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
FREEZE_TRANSFORMER = False
SEED = 42

set_seed(SEED)

In [18]:
# --- INITIALIZE DATA ---
train_examples = read_jsonl(f"{DATA_DIR}/train.jsonl")
dev_examples = read_jsonl(f"{DATA_DIR}/dev.jsonl")
test_examples = read_jsonl(f"{DATA_DIR}/test.jsonl")

word_vocab, char_vocab = build_vocabs(train_examples)
span_labels = sorted({label.label for ex in train_examples for label in ex.labels})
labels = build_sequence_labels(span_labels, scheme="bilou")
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_dataset = BiLSTMCRFDataset(train_examples, word_vocab, char_vocab, label2id, tokenizer, "bilou")
dev_dataset = BiLSTMCRFDataset(dev_examples, word_vocab, char_vocab, label2id, tokenizer, "bilou", is_train=False)
test_dataset = BiLSTMCRFDataset(test_examples, word_vocab, char_vocab, label2id, tokenizer, "bilou", is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Data loaded! Train/Dev/Test size: {len(train_dataset)} / {len(dev_dataset)} / {len(test_dataset)}")

# --- Compute sqrt-balanced class weights ---
label_freq = Counter()
for feat in train_dataset.features:
    for l in feat['labels']:
        label_freq[l] += 1

total_tokens = sum(label_freq.values())
n_classes = len(label2id)
class_weights = torch.ones(n_classes)
for label_id, count in label_freq.items():
    if count > 0:
        class_weights[label_id] = np.sqrt(total_tokens / (n_classes * count))

# Normalize: O tag weight = 1.0
class_weights = class_weights / class_weights[label2id["O"]]
print(f"Class weights computed (min={class_weights.min():.2f}, max={class_weights.max():.2f})")
print(f"Scheme: BILOU | Labels: {len(labels)}")


# --- Save Normalized Data ---
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
write_jsonl(f"{OUTPUT_DIR}/train_normalized.jsonl", train_examples)
write_jsonl(f"{OUTPUT_DIR}/dev_normalized.jsonl", dev_examples)
write_jsonl(f"{OUTPUT_DIR}/test_normalized.jsonl", test_examples)
print("Normalized datasets saved.")


Data loaded! Train/Dev/Test size: 7785 / 1112 / 2225
Class weights computed (min=1.00, max=354.24)
Scheme: BILOU | Labels: 121
Normalized datasets saved.


In [19]:
# --- TRAINING LOOP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


model = BiLSTM_CRF_ABSA(
    vocab_size=len(word_vocab),
    char_vocab_size=len(char_vocab),
    num_labels=len(labels),
    transformer_name=MODEL_NAME,
    freeze_transformer=FREEZE_TRANSFORMER
).to(device)

# --- Differential Learning Rates ---
transformer_params = []
other_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "transformer" in name:
        transformer_params.append(param)
    else:
        other_params.append(param)

# Ép XLM-R học siêu chậm (2e-5) để bảo toàn kiến thức cũ
# Cho lớp BiLSTM-CRF học nhanh (LEARNING_RATE = 1e-3)
optimizer = torch.optim.AdamW([
    {"params": transformer_params, "lr": 2e-5},
    {"params": other_params, "lr": 1e-3}  # Hoặc dùng biến LEARNING_RATE
])

from transformers import get_linear_schedule_with_warmup
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps) # Dành ra 10% số bước đầu tiên để khởi động từ từ

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

best_f1 = 0.0
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        
        tags = batch.pop("tags").clone()
        valid_mask = (tags != -100)
        tags[~valid_mask] = 0 
        batch["tags"] = tags
        
        loss = model(**batch)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        
    # Validation (span-based)
    model.eval()
    pred_labels = []
    with torch.no_grad():
        for batch in dev_loader:
            batch_device = {k: v.to(device) for k, v in batch.items()}
            tags_dev = batch_device.pop("tags")
            best_paths = model(**batch_device)
            
            tags = tags_dev.cpu().numpy()
            padded_paths = []
            for p, t in zip(best_paths, tags):
                p_len, t_len = len(p), len(t)
                if p_len < t_len: p.extend([0]*(t_len - p_len))
                padded_paths.append(p[:t_len])
            pred_labels.extend(padded_paths)
            
    metrics = compute_span_metrics(dev_examples, pred_labels, dev_dataset.all_token_spans, id2label)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f} | Dev Span-F1: {metrics['f1']:.4f} (Prec: {metrics['precision']:.4f}, Rec: {metrics['recall']:.4f})")
    
    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        torch.save(model.state_dict(), Path(OUTPUT_DIR) / "best_model.pt")


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/30 - Loss: 93.5915 | Dev Span-F1: 0.3547 (Prec: 0.3156, Rec: 0.4048)
Epoch 2/30 - Loss: 39.9879 | Dev Span-F1: 0.4843 (Prec: 0.4536, Rec: 0.5195)
Epoch 3/30 - Loss: 28.5132 | Dev Span-F1: 0.5353 (Prec: 0.5032, Rec: 0.5717)
Epoch 4/30 - Loss: 20.2378 | Dev Span-F1: 0.5806 (Prec: 0.5732, Rec: 0.5882)
Epoch 5/30 - Loss: 14.4703 | Dev Span-F1: 0.5891 (Prec: 0.5766, Rec: 0.6022)
Epoch 6/30 - Loss: 10.5398 | Dev Span-F1: 0.5909 (Prec: 0.5800, Rec: 0.6022)
Epoch 7/30 - Loss: 8.0029 | Dev Span-F1: 0.6027 (Prec: 0.6026, Rec: 0.6027)
Epoch 8/30 - Loss: 6.1831 | Dev Span-F1: 0.6098 (Prec: 0.6056, Rec: 0.6142)
Epoch 9/30 - Loss: 4.8631 | Dev Span-F1: 0.6048 (Prec: 0.5970, Rec: 0.6128)
Epoch 10/30 - Loss: 4.0350 | Dev Span-F1: 0.6076 (Prec: 0.6068, Rec: 0.6083)
Epoch 11/30 - Loss: 3.3782 | Dev Span-F1: 0.6066 (Prec: 0.5979, Rec: 0.6156)
Epoch 12/30 - Loss: 2.7777 | Dev Span-F1: 0.6073 (Prec: 0.6010, Rec: 0.6136)
Epoch 13/30 - Loss: 2.2887 | Dev Span-F1: 0.6061 (Prec: 0.6031, Rec: 0.6092)
Ep

In [20]:
# --- EVALUATION ON TEST SET (Span-based) ---
print("\n--- Training Completed. Evaluating on Test Set ---")
model.load_state_dict(torch.load(Path(OUTPUT_DIR) / "best_model.pt"))
model.eval()

test_pred_labels = []
with torch.no_grad():
    for batch in test_loader:
        batch_device = {k: v.to(device) for k, v in batch.items()}
        tags_test = batch_device.pop("tags")
        best_paths = model(**batch_device)
        
        tags = tags_test.cpu().numpy()
        padded_paths = []
        for p, t in zip(best_paths, tags):
            p_len, t_len = len(p), len(t)
            if p_len < t_len: p.extend([0]*(t_len - p_len))
            padded_paths.append(p[:t_len])
        test_pred_labels.extend(padded_paths)

# --- Span-based Evaluation ---
all_gold_spans = []
all_pred_spans = []
for ex, pred_tags, tok_spans in zip(test_examples, test_pred_labels, test_dataset.all_token_spans):
    gold = {(l.start, l.end, l.label) for l in ex.labels}
    pred = bio_tags_to_spans(pred_tags, tok_spans, id2label)
    all_gold_spans.append(gold)
    all_pred_spans.append(pred)

# Overall micro metrics
total_tp = sum(len(g & p) for g, p in zip(all_gold_spans, all_pred_spans))
total_pred = sum(len(p) for p in all_pred_spans)
total_gold = sum(len(g) for g in all_gold_spans)
precision = total_tp / total_pred if total_pred > 0 else 0.0
recall = total_tp / total_gold if total_gold > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n{'='*55}")
print(f"  SPAN-BASED EVALUATION (Exact Match)")
print(f"{'='*55}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  (TP={total_tp}, Pred={total_pred}, Gold={total_gold})")

# Per-label breakdown
label_stats = defaultdict(lambda: {"tp": 0, "pred": 0, "gold": 0})

for gold_set, pred_set in zip(all_gold_spans, all_pred_spans):
    for s, e, label in gold_set:
        label_stats[label]["gold"] += 1
        if (s, e, label) in pred_set:
            label_stats[label]["tp"] += 1
    for s, e, label in pred_set:
        label_stats[label]["pred"] += 1

print(f"\n{'='*55}")
print(f"  PER-LABEL BREAKDOWN")
print(f"{'='*55}")
print(f"  {'Label':<25} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Gold':>5} {'Pred':>5} {'TP':>5}")
print(f"  {'-'*67}")

macro_p_sum = 0
macro_r_sum = 0
macro_f1_sum = 0
num_labels = len(label_stats)

for label in sorted(label_stats.keys()):
    stats = label_stats[label]
    p = stats["tp"] / stats["pred"] if stats["pred"] > 0 else 0
    r = stats["tp"] / stats["gold"] if stats["gold"] > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    macro_p_sum += p
    macro_r_sum += r
    macro_f1_sum += f
    print(f"  {label:<25} {p:>7.4f} {r:>7.4f} {f:>7.4f} {stats['gold']:>5} {stats['pred']:>5} {stats['tp']:>5}")

macro_p = macro_p_sum / num_labels if num_labels > 0 else 0
macro_r = macro_r_sum / num_labels if num_labels > 0 else 0
macro_f1 = macro_f1_sum / num_labels if num_labels > 0 else 0

print(f"\n{'='*55}")
print(f"  MACRO METRICS")
print(f"{'='*55}")
print(f"  Macro Precision: {macro_p:.4f}")
print(f"  Macro Recall:    {macro_r:.4f}")
print(f"  Macro F1:        {macro_f1:.4f}")

# --- Save Predictions ---
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
pred_rows = []
for ex, pred_set in zip(test_examples, all_pred_spans):
    labels = [[s, e, l] for s, e, l in pred_set]
    pred_rows.append({"text": ex.text, "labels": labels})

with open(f"{OUTPUT_DIR}/test_predictions.jsonl", "w", encoding="utf-8") as f:
    for row in pred_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to {OUTPUT_DIR}/test_predictions.jsonl")



--- Training Completed. Evaluating on Test Set ---

  SPAN-BASED EVALUATION (Exact Match)
  Precision: 0.6108
  Recall:    0.6192
  F1:        0.6150
  (TP=4361, Pred=7140, Gold=7043)

  PER-LABEL BREAKDOWN
  Label                        Prec     Rec      F1  Gold  Pred    TP
  -------------------------------------------------------------------
  BATTERY#NEGATIVE           0.6026  0.5743  0.5881   404   385   232
  BATTERY#NEUTRAL            0.3061  0.4348  0.3593    69    98    30
  BATTERY#POSITIVE           0.8029  0.7816  0.7921   641   624   501
  CAMERA#NEGATIVE            0.5667  0.6398  0.6010   186   210   119
  CAMERA#NEUTRAL             0.5571  0.5000  0.5270    78    70    39
  CAMERA#POSITIVE            0.7688  0.7535  0.7611   353   346   266
  DESIGN#NEGATIVE            0.3548  0.4536  0.3982    97   124    44
  DESIGN#NEUTRAL             0.3000  0.2000  0.2400    15    10     3
  DESIGN#POSITIVE            0.7409  0.7483  0.7446   298   301   223
  FEATURES#NEGATIVE   